## Fase 6B: Extração de Informações Relevantes para o Modelo de IA (Parte 2)

In [ ]:
# Configuração do ambiente
import sys
sys.path.append('../../')
import setup_notebook
setup_notebook.setup_environment()

%matplotlib inline

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import os
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from scipy import stats

# Configurar o estilo dos gráficos
plt.style.use('fivethirtyeight')
sns.set(style="whitegrid")

## 1. Carregamento dos Dados

In [ ]:
# Carregar os dados de NDVI/EVI
df_ndvi_nf = pd.read_csv('../../assets/ndvi_mensal_nova_friburgo.csv')
df_ndvi_t = pd.read_csv('../../assets/ndvi_mensal_teresopolis.csv')

# Carregar os dados de produtividade agrícola
df_prod_combinados = pd.read_csv('../../assets/dados_produtividade_combinados.csv')

# Filtrar os dados de produtividade para o ano de 2017
df_prod_2017 = df_prod_combinados[df_prod_combinados['Ano'] == 2017]

# Agrupar os dados de produtividade por município e tipo de lavoura
df_prod_grouped = df_prod_2017.groupby(['Município', 'Tipo de Lavoura'])['Produtividade (t/ha)'].mean().reset_index()

# Calcular a média de produtividade por município
df_prod_mean = df_prod_grouped.groupby('Município')['Produtividade (t/ha)'].mean().reset_index()

# Exibir os dados de produtividade
print("Produtividade média por município em 2017:")
print(df_prod_mean)

In [ ]:
# Converter a coluna de data para o formato datetime
df_ndvi_nf['Data'] = pd.to_datetime(df_ndvi_nf['Ano'].astype(str) + '-' + df_ndvi_nf['Mês'].astype(str) + '-01')
df_ndvi_t['Data'] = pd.to_datetime(df_ndvi_t['Ano'].astype(str) + '-' + df_ndvi_t['Mês'].astype(str) + '-01')

# Filtrar os dados para o ano de 2017
df_ndvi_nf_2017 = df_ndvi_nf[df_ndvi_nf['Ano'] == 2017].copy()
df_ndvi_t_2017 = df_ndvi_t[df_ndvi_t['Ano'] == 2017].copy()

# Adicionar coluna de município
df_ndvi_nf_2017['Município'] = 'Nova Friburgo'
df_ndvi_t_2017['Município'] = 'Teresópolis'

# Concatenar os dataframes
df_ndvi_2017 = pd.concat([df_ndvi_nf_2017, df_ndvi_t_2017])

# Exibir as primeiras linhas do dataframe combinado
print("Primeiras linhas do dataframe combinado para 2017:")
print(df_ndvi_2017[['Município', 'Ano', 'Mês', 'Data', 'EVI_mean', 'SG_mean']].head())

## 2. Identificação dos Períodos Críticos de Crescimento da Cultura

In [ ]:
# Pivotar o dataframe para ter uma linha por município e colunas para cada mês
df_monthly_pivot = df_ndvi_2017.pivot_table(
    index='Município',
    columns='Mês',
    values=['EVI_mean', 'SG_mean']
).reset_index()

# Renomear as colunas
df_monthly_pivot.columns = ['Município'] + [
    f"{col[0]}_mes_{col[1]}" for col in df_monthly_pivot.columns.values[1:]
]

# Mesclar com os dados de produtividade
df_monthly_features = pd.merge(df_monthly_pivot, df_prod_mean, on='Município')

# Exibir o dataframe com as features mensais e a produtividade
print("Dataframe com features mensais e produtividade para 2017:")
print(df_monthly_features)

In [ ]:
# Carregar os dados pré-processados
df_ndvi_nf_prep = pd.read_csv('../../sprint2/assets/ndvi_mensal_nova_friburgo_preprocessado.csv')
df_ndvi_t_prep = pd.read_csv('../../sprint2/assets/ndvi_mensal_teresopolis_preprocessado.csv')

# Filtrar os dados para o ano de 2017 (Ano = 0.68 nos dados pré-processados)
df_ndvi_nf_2017_prep = df_ndvi_nf_prep[df_ndvi_nf_prep['Ano'] == 0.6800000000000068].copy()
df_ndvi_t_2017_prep = df_ndvi_t_prep[df_ndvi_t_prep['Ano'] == 0.6800000000000068].copy()

# Adicionar coluna de município
df_ndvi_nf_2017_prep['Município'] = 'Nova Friburgo'
df_ndvi_t_2017_prep['Município'] = 'Teresópolis'

# Concatenar os dataframes
df_ndvi_2017_prep = pd.concat([df_ndvi_nf_2017_prep, df_ndvi_t_2017_prep])

# Exibir as primeiras linhas do dataframe combinado
print("Primeiras linhas do dataframe pré-processado combinado para 2017:")
print(df_ndvi_2017_prep[['Município', 'Mês', 'EVI_mean', 'SG_mean']].head())

In [ ]:
# Criar um dataframe com as diferenças relativas entre os índices e a produtividade
# Primeiro, vamos calcular a diferença relativa na produtividade entre os dois municípios
prod_nf = df_monthly_features[df_monthly_features['Município'] == 'Nova Friburgo']['Produtividade (t/ha)'].values[0]
prod_t = df_monthly_features[df_monthly_features['Município'] == 'Teresópolis']['Produtividade (t/ha)'].values[0]
prod_diff = (prod_nf - prod_t) / ((prod_nf + prod_t) / 2)  # Diferença relativa

# Agora, vamos calcular as diferenças relativas para cada índice mensal
feature_diffs = {}
for col in df_monthly_features.columns:
    if col not in ['Município', 'Produtividade (t/ha)']:
        val_nf = df_monthly_features[df_monthly_features['Município'] == 'Nova Friburgo'][col].values[0]
        val_t = df_monthly_features[df_monthly_features['Município'] == 'Teresópolis'][col].values[0]
        if not pd.isna(val_nf) and not pd.isna(val_t) and val_nf != 0 and val_t != 0:
            diff = (val_nf - val_t) / ((val_nf + val_t) / 2)  # Diferença relativa
            # Calcular a similaridade com a diferença na produtividade
            similarity = 1 - abs(diff - prod_diff) / (abs(diff) + abs(prod_diff) + 1e-10)
            feature_diffs[col] = {
                'Diferença Relativa': diff,
                'Similaridade com Produtividade': similarity
            }

# Criar dataframe com as diferenças
df_diffs = pd.DataFrame.from_dict(feature_diffs, orient='index')

# Ordenar por similaridade com a produtividade
df_diffs = df_diffs.sort_values('Similaridade com Produtividade', ascending=False)

# Exibir as diferenças
print("Similaridade entre as diferenças relativas dos índices e da produtividade:")
print(df_diffs.head(10))

In [ ]:
# Vamos visualizar as similaridades em um gráfico de barras
plt.figure(figsize=(12, 8))

# Selecionar as 10 features com maior similaridade
top_features = df_diffs.head(10).index

# Plotar o gráfico de barras
df_diffs.loc[top_features, 'Similaridade com Produtividade'].plot(kind='bar', figsize=(12, 8))
plt.title('Similaridade entre as Diferenças Relativas dos Índices e da Produtividade')
plt.xlabel('Índice Mensal')
plt.ylabel('Similaridade')
plt.xticks(rotation=45, ha='right')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Vamos extrair o mês de cada feature
df_diffs['Mês'] = df_diffs.index.str.extract(r'mes_(\d+)').astype(float)

# Filtrar apenas as features de EVI
df_diffs_evi = df_diffs[df_diffs.index.str.startswith('EVI_mean')]

# Ordenar por mês
df_diffs_evi = df_diffs_evi.sort_values('Mês')

# Plotar a similaridade por mês para EVI
plt.figure(figsize=(12, 6))
plt.plot(df_diffs_evi['Mês'], df_diffs_evi['Similaridade com Produtividade'], 'g-', marker='o', linewidth=2, label='Similaridade')
plt.title('Similaridade entre EVI Mensal e Produtividade')
plt.xlabel('Mês')
plt.ylabel('Similaridade')
plt.xticks(range(1, 13), ['Jan', 'Fev', 'Mar', 'Abr', 'Mai', 'Jun', 'Jul', 'Ago', 'Set', 'Out', 'Nov', 'Dez'])
plt.axhline(y=0.5, color='r', linestyle='-', alpha=0.3)
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Filtrar apenas as features de SG
df_diffs_sg = df_diffs[df_diffs.index.str.startswith('SG_mean')]

# Ordenar por mês
df_diffs_sg = df_diffs_sg.sort_values('Mês')

# Plotar a similaridade por mês para SG
plt.figure(figsize=(12, 6))
plt.plot(df_diffs_sg['Mês'], df_diffs_sg['Similaridade com Produtividade'], 'b-', marker='o', linewidth=2, label='Similaridade')
plt.title('Similaridade entre Savitzky-Golay Mensal e Produtividade')
plt.xlabel('Mês')
plt.ylabel('Similaridade')
plt.xticks(range(1, 13), ['Jan', 'Fev', 'Mar', 'Abr', 'Mai', 'Jun', 'Jul', 'Ago', 'Set', 'Out', 'Nov', 'Dez'])
plt.axhline(y=0.5, color='r', linestyle='-', alpha=0.3)
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## 3. Extração de Features Temporais

In [ ]:
# Selecionar as 5 features com maior similaridade
top_5_features = df_diffs.head(5).index

# Criar um dataframe com as features selecionadas
df_selected_features = df_monthly_features[['Município', 'Produtividade (t/ha)'] + list(top_5_features)]

# Exibir o dataframe com as features selecionadas
print("Dataframe com as features selecionadas:")
print(df_selected_features)

In [ ]:
# Identificar os meses críticos com base na análise de similaridade
# Definir meses críticos manualmente para evitar erros
critical_months = [2, 7, 9]  # Fevereiro, Julho, Setembro

print(f"Meses críticos identificados: {critical_months}")

# Calcular features para os períodos críticos
critical_features = {}

for municipio in ['Nova Friburgo', 'Teresópolis']:
    # Filtrar os dados para o município e os meses críticos
    df_critical = df_ndvi_2017[(df_ndvi_2017['Município'] == municipio) & 
                              (df_ndvi_2017['Mês'].isin(critical_months))]
    
    # Calcular estatísticas para os períodos críticos
    critical_features[municipio] = {
        'EVI_critical_mean': df_critical['EVI_mean'].mean(),
        'EVI_critical_min': df_critical['EVI_mean'].min(),
        'EVI_critical_max': df_critical['EVI_mean'].max(),
        'EVI_critical_std': df_critical['EVI_mean'].std(),
        'SG_critical_mean': df_critical['SG_mean'].mean(),
        'SG_critical_min': df_critical['SG_mean'].min(),
        'SG_critical_max': df_critical['SG_mean'].max(),
        'SG_critical_std': df_critical['SG_mean'].std()
    }

# Criar um dataframe com as features dos períodos críticos
df_critical_features = pd.DataFrame.from_dict(critical_features, orient='index').reset_index()
df_critical_features.rename(columns={'index': 'Município'}, inplace=True)

# Exibir o dataframe com as features dos períodos críticos
print("\nDataframe com as features dos períodos críticos:")
print(df_critical_features)

In [ ]:
# Mesclar com as features selecionadas
df_final_features = pd.merge(df_selected_features, df_critical_features, on='Município')

# Exibir o dataframe com as features finais
print("Dataframe com as features finais:")
print(df_final_features)

## 4. Seleção das Features Mais Relevantes

In [ ]:
# Calcular a diferença relativa na produtividade entre os dois municípios
prod_nf = df_final_features[df_final_features['Município'] == 'Nova Friburgo']['Produtividade (t/ha)'].values[0]
prod_t = df_final_features[df_final_features['Município'] == 'Teresópolis']['Produtividade (t/ha)'].values[0]
prod_diff = (prod_nf - prod_t) / ((prod_nf + prod_t) / 2)  # Diferença relativa

# Calcular as diferenças relativas para cada feature final
feature_diffs_final = {}
for col in df_final_features.columns:
    if col not in ['Município', 'Produtividade (t/ha)']:
        val_nf = df_final_features[df_final_features['Município'] == 'Nova Friburgo'][col].values[0]
        val_t = df_final_features[df_final_features['Município'] == 'Teresópolis'][col].values[0]
        if not pd.isna(val_nf) and not pd.isna(val_t) and val_nf != 0 and val_t != 0:
            diff = (val_nf - val_t) / ((val_nf + val_t) / 2)  # Diferença relativa
            # Calcular a similaridade com a diferença na produtividade
            similarity = 1 - abs(diff - prod_diff) / (abs(diff) + abs(prod_diff) + 1e-10)
            feature_diffs_final[col] = {
                'Diferença Relativa': diff,
                'Similaridade com Produtividade': similarity
            }

# Criar dataframe com as diferenças finais
df_diffs_final = pd.DataFrame.from_dict(feature_diffs_final, orient='index')

# Ordenar por similaridade com a produtividade
df_diffs_final = df_diffs_final.sort_values('Similaridade com Produtividade', ascending=False)

# Exibir as similaridades
print("Similaridade entre as features finais e a produtividade:")
print(df_diffs_final)

In [ ]:
# Vamos visualizar as similaridades em um gráfico de barras
plt.figure(figsize=(12, 8))
df_diffs_final['Similaridade com Produtividade'].plot(kind='bar', figsize=(12, 8))
plt.title('Similaridade entre as Features Finais e a Produtividade')
plt.xlabel('Feature')
plt.ylabel('Similaridade')
plt.xticks(rotation=45, ha='right')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Vamos selecionar as features mais relevantes com base na similaridade
# Vamos considerar as features com similaridade acima de 0.5
relevant_features = df_diffs_final[df_diffs_final['Similaridade com Produtividade'] > 0.5].index.tolist()

# Exibir as features relevantes
print(f"Features relevantes selecionadas: {relevant_features}")

# Criar um dataframe com as features relevantes
df_relevant_features = df_final_features[['Município', 'Produtividade (t/ha)'] + relevant_features]

# Exibir o dataframe com as features relevantes
print("\nDataframe com as features relevantes:")
print(df_relevant_features)

## 5. Exportação das Features para o Modelo de IA

In [ ]:
# Criar diretório para os dados processados
os.makedirs('../../sprint2/assets', exist_ok=True)

# Exportar as features finais
df_final_features.to_csv('../../sprint2/assets/features_finais.csv', index=False)

# Exportar as features relevantes
df_relevant_features.to_csv('../../sprint2/assets/features_relevantes.csv', index=False)

# Exportar a lista de meses críticos
pd.DataFrame({'Mês Crítico': critical_months}).to_csv('../../sprint2/assets/meses_criticos.csv', index=False)

# Exportar as similaridades das features
df_diffs_final.to_csv('../../sprint2/assets/similaridades_features.csv')

print("Features exportadas com sucesso!")